In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_analytics"
GOLD = "gold"

checks = []

def add_check(check_name, status, actual, expected, details):
    checks.append(
        (
            check_name,
            status,
            str(actual),
            str(expected),
            details
        )
    )

# ============================================================
# DIMENSION KEY QUALITY
# ============================================================

dimension_checks = [
    ("dim_patient", "patient_id"),
    ("dim_provider", "provider_id"),
    ("dim_facility", "organization_id"),
    ("dim_payer", "payer_id")
]

for table_name, key_col in dimension_checks:

    df = spark.table(
        f"{CATALOG}.{GOLD}.{table_name}"
    )

    total_rows = df.count()

    null_keys = (
        df.filter(
            F.col(key_col).isNull()
        ).count()
    )

    distinct_keys = (
        df.select(key_col)
        .distinct()
        .count()
    )

    status = (
        "PASS"
        if null_keys == 0
        and distinct_keys == total_rows
        else "FAIL"
    )

    add_check(
        f"{table_name} key integrity",
        status,
        f"rows={total_rows}, nulls={null_keys}, distinct={distinct_keys}",
        "0 nulls and unique keys",
        key_col
    )


# ============================================================
# FACT ENCOUNTER UNIQUENESS
# ============================================================

enc = spark.table(
    f"{CATALOG}.{GOLD}.fact_encounter"
)

enc_rows = enc.count()

enc_distinct = (
    enc.select("encounter_id")
    .distinct()
    .count()
)

enc_nulls = (
    enc.filter(
        F.col("encounter_id").isNull()
    ).count()
)

add_check(
    "fact_encounter encounter_id integrity",
    (
        "PASS"
        if enc_rows == enc_distinct
        and enc_nulls == 0
        else "FAIL"
    ),
    f"rows={enc_rows}, distinct={enc_distinct}, nulls={enc_nulls}",
    "unique and non-null",
    "Encounter grain validation"
)


# ============================================================
# REFERENTIAL INTEGRITY
# ============================================================

relationships = [
    (
        "patient_id",
        "dim_patient",
        "patient_id"
    ),
    (
        "provider_id",
        "dim_provider",
        "provider_id"
    ),
    (
        "organization_id",
        "dim_facility",
        "organization_id"
    ),
    (
        "payer_id",
        "dim_payer",
        "payer_id"
    )
]

for fact_col, dim_table, dim_col in relationships:

    dim = spark.table(
        f"{CATALOG}.{GOLD}.{dim_table}"
    )

    orphan_count = (
        enc
        .filter(
            F.col(fact_col).isNotNull()
        )
        .select(fact_col)
        .distinct()
        .join(
            dim.select(
                F.col(dim_col).alias(fact_col)
            ).distinct(),
            fact_col,
            "left_anti"
        )
        .count()
    )

    add_check(
        f"fact_encounter → {dim_table}",
        "PASS" if orphan_count == 0 else "FAIL",
        orphan_count,
        0,
        f"Unmatched {fact_col} values"
    )


# ============================================================
# FINANCIAL SANITY
# ============================================================

negative_financials = (
    enc.filter(
        (F.col("total_claim_cost") < 0)
        | (F.col("payer_coverage") < 0)
        | (F.col("patient_responsibility") < 0)
    )
    .count()
)

add_check(
    "Encounter financial values non-negative",
    "PASS" if negative_financials == 0 else "FAIL",
    negative_financials,
    0,
    "Negative claim/payer/patient amounts"
)


# ============================================================
# READMISSION LOGIC VALIDATION
# ============================================================

readm = spark.table(
    f"{CATALOG}.{GOLD}.fact_readmission"
)

invalid_flags = (
    readm.filter(
        F.col("readmission_30d_flag").isNull()
        | (~F.col("readmission_30d_flag").isin(0, 1))
    )
    .count()
)

add_check(
    "Readmission flag domain",
    "PASS" if invalid_flags == 0 else "FAIL",
    invalid_flags,
    0,
    "Flag must be 0 or 1"
)

invalid_readmission_window = (
    readm.filter(
        (F.col("readmission_30d_flag") == 1)
        &
        (
            F.col("days_to_readmission").isNull()
            | (F.col("days_to_readmission") < 0)
            | (F.col("days_to_readmission") > 30)
        )
    )
    .count()
)

add_check(
    "30-day readmission window",
    (
        "PASS"
        if invalid_readmission_window == 0
        else "FAIL"
    ),
    invalid_readmission_window,
    0,
    "Flagged readmissions must occur within 0-30 days"
)


# ============================================================
# EXECUTIVE MART GRAIN
# ============================================================

exec_mart = spark.table(
    f"{CATALOG}.{GOLD}.mart_executive_healthcare"
)

exec_rows = exec_mart.count()

add_check(
    "Executive mart grain",
    "PASS" if exec_rows == 1 else "FAIL",
    exec_rows,
    1,
    "Executive KPI mart should contain one aggregate row"
)


# ============================================================
# OUTPUT
# ============================================================

qa_df = spark.createDataFrame(
    checks,
    [
        "check_name",
        "status",
        "actual",
        "expected",
        "details"
    ]
)

display(qa_df.orderBy("status", "check_name"))

print(
    "Checks:",
    qa_df.count()
)

print(
    "Failures:",
    qa_df.filter(
        F.col("status") == "FAIL"
    ).count()
)

check_name,status,actual,expected,details
30-day readmission window,PASS,0,0,Flagged readmissions must occur within 0-30 days
Encounter financial values non-negative,PASS,0,0,Negative claim/payer/patient amounts
Executive mart grain,PASS,1,1,Executive KPI mart should contain one aggregate row
Readmission flag domain,PASS,0,0,Flag must be 0 or 1
dim_facility key integrity,PASS,"rows=827, nulls=0, distinct=827",0 nulls and unique keys,organization_id
dim_patient key integrity,PASS,"rows=1138, nulls=0, distinct=1138",0 nulls and unique keys,patient_id
dim_payer key integrity,PASS,"rows=10, nulls=0, distinct=10",0 nulls and unique keys,payer_id
dim_provider key integrity,PASS,"rows=827, nulls=0, distinct=827",0 nulls and unique keys,provider_id
fact_encounter encounter_id integrity,PASS,"rows=63795, distinct=63795, nulls=0",unique and non-null,Encounter grain validation
fact_encounter → dim_facility,PASS,0,0,Unmatched organization_id values


Checks: 13
Failures: 0
